# form-correction: cloud runner

Cloud entry point. Cells follow ARCHITECTURE.md section 6.

## 1. Setup
Install dependencies and set deterministic seeds.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/wizwall-anwar/form-correction.git"
if os.path.isdir("/kaggle"):
    REPO_DIR = "/kaggle/working/form-correction"
elif os.path.isdir("/content"):
    REPO_DIR = "/content/form-correction"
else:
    REPO_DIR = os.getcwd()

try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        get_ipython().system(f"git -C {REPO_DIR} pull --ff-only")
    elif not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

try:
    get_ipython().system("pip install -q -r requirements.txt")
except Exception:
    pass

import random
import numpy as np
from src.config import RANDOM_SEED
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"[Setup] done (REPO_DIR={REPO_DIR!r})")

## 2. Config
Set `DATA_DIR` to a manual dataset path to skip fetching.

In [ ]:
import os
from src.config import resolve_data_dir

# --- Gemini API key for VLM agreement (cell 6) ---
# Option 1: Colab Secrets sidebar > key icon > add GEMINI_API_KEY
# Option 2: Kaggle > Add-ons > Secrets > add GEMINI_API_KEY
try:
    from google.colab import userdata
    _key = userdata.get('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
        os.environ['VLM_BACKEND'] = 'gemini'
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
        os.environ['VLM_BACKEND'] = 'gemini'
except Exception:
    pass

if os.environ.get('GEMINI_API_KEY'):
    print(f"[Config] Gemini API key set ({len(os.environ['GEMINI_API_KEY'])} chars)")
else:
    print('[Config] No Gemini API key; VLM agreement will use offline stub')

DATA_DIR = resolve_data_dir(None)
print(f'[Config] DATA_DIR={DATA_DIR!r}' if DATA_DIR else '[Config] no local dataset found; synthetic fallback will run')


## 3. Data
Load through the bounded fallback chain, using synthetic squats if loading fails.

In [ ]:
from src import m00_data
data_is_synthetic = False
try:
    samples = m00_data.load_data(DATA_DIR)
    data_source = samples[0].data_source if samples else "synthetic"
    data_is_synthetic = data_source == "synthetic"
except Exception:
    samples = m00_data.make_synthetic_squats(20, RANDOM_SEED)
    data_source = "synthetic"
    data_is_synthetic = True
print(f"[Data] {data_source} ({len(samples)} samples)")

## 4. Features
Create rotated, jittered synthetic views and compute kinematic flags.

In [ ]:
from src.config import JITTER_SIGMA, ROTATION_ANGLES_DEG
from src.m01_features import add_jitter, kinematic_flags, rotate_view
views_by_angle = {}
flags_by_angle = {}
try:
    for angle in ROTATION_ANGLES_DEG:
        views_by_angle[angle] = []
        flags_by_angle[angle] = []
        for index, sample in enumerate(samples):
            joints = add_jitter(rotate_view(sample.joints, angle), JITTER_SIGMA, RANDOM_SEED + index + int(angle))
            views_by_angle[angle].append(joints)
            flags_by_angle[angle].append(kinematic_flags(joints))
except Exception:
    views_by_angle = {0: [sample.joints for sample in samples]}
    flags_by_angle = {0: [kinematic_flags(sample.joints) for sample in samples]}
print(f"[Features] {len(flags_by_angle)} view angles")

## 5. Detector
Run splits, baseline training, calibration, abstention, and metrics.

In [ ]:
from src import m02_detector
from src.m01_features import flag_error

true_bins = [int(sample.clinician_bin) for sample in samples]
predictions = []
risk = {'coverage': [], 'risk': []}
detector_metrics = {'macro_f1': 0.0, 'accuracy': 0.0, 'ece': 0.0}
clip_metrics = {'accuracy': 0.0}
table2_rows = []
try:
    train_samples, test_samples = m02_detector.make_splits(samples, RANDOM_SEED)
    clip_train, clip_test = m02_detector.make_clip_split(samples, RANDOM_SEED)
    model = m02_detector.train_baseline(train_samples, RANDOM_SEED)
    calibrated = m02_detector.calibrate(model, test_samples or train_samples)
    predictions = m02_detector.predict_with_abstention(calibrated, samples)
    risk = m02_detector.risk_coverage(predictions, true_bins)
    detector_metrics = m02_detector.evaluate(predictions, true_bins)
    clip_model = m02_detector.train_baseline(clip_train, RANDOM_SEED)
    clip_predictions = m02_detector.predict_with_abstention(clip_model, clip_test)
    clip_metrics = m02_detector.evaluate(clip_predictions, [int(s.clinician_bin) for s in clip_test])

    # --- Per-angle evaluation for real table2 ---
    ref_flags = flags_by_angle.get(0, [])
    feat_keys = ('knee_flexion_max_deg', 'trunk_lean_max_deg', 'symmetry_diff_deg')
    for angle in sorted(flags_by_angle):
        angle_flags = flags_by_angle[angle]
        angle_feats = np.asarray([[f[k] for k in feat_keys] for f in angle_flags], dtype=float).reshape(-1, 3)
        angle_proba = calibrated.predict_proba(angle_feats)
        angle_classes = np.asarray(calibrated.classes_)
        angle_preds = [{'pred_bin': int(angle_classes[int(np.argmax(p))]),
                        'confidence': float(np.max(p)),
                        'abstain_flag': bool(np.max(p) < 0.70)} for p in angle_proba]
        angle_eval = m02_detector.evaluate(angle_preds, true_bins)
        errors = [flag_error(vf, rf) for vf, rf in zip(angle_flags, ref_flags)] if ref_flags else []
        mean_errs = {k: float(np.mean([e[k] for e in errors])) for k in feat_keys} if errors else {}
        table2_rows.append({'angle': angle,
                            'accuracy': angle_eval.get('accuracy', 0.0),
                            'macro_f1': angle_eval.get('macro_f1', 0.0),
                            'ece': angle_eval.get('ece', 0.0),
                            'flag_error': mean_errs})
    print(f'[Detector] per-angle: {[r["angle"] for r in table2_rows]}')
except Exception as exc:
    predictions = [{'pred_bin': lb, 'confidence': 1.0, 'abstain_flag': False} for lb in true_bins]
    print(f'[Detector] fallback: {exc}')

# Build detector records for the demo
detector_records = []
base_flags = flags_by_angle.get(0, [])
for index, sample in enumerate(samples):
    flags = base_flags[index] if index < len(base_flags) else m02_detector.kinematic_flags(sample.joints)
    pred = predictions[index] if index < len(predictions) else {'pred_bin': int(sample.clinician_bin), 'confidence': 1.0, 'abstain_flag': False}
    detector_records.append({'flags': flags, 'view_deg': 0.0, **pred})
print(f'[Detector] {len(detector_records)} records, subj-acc={detector_metrics.get("accuracy",0):.3f}, clip-acc={clip_metrics.get("accuracy",0):.3f}')


## 6. Agreement
Compare the offline VLM stub with clinician bins at 0 and 45 degrees.

In [ ]:
from src.m03_agreement import agreement_report
try:
    backend = os.environ.get('VLM_BACKEND', 'placeholder')
    print(f'[Agreement] backend={backend}')
    agreement = agreement_report(samples, view_degs=(0, 45), seed=RANDOM_SEED, backend=backend)
    for deg, metrics in agreement.get('model_vs_clinician_reliability', {}).items():
        n = metrics.get('sample_count', 0)
        kappa = metrics.get('cohen_kappa', 0.0)
        qw = metrics.get('quadratic_weighted_cohen_kappa', 0.0)
        print(f'  {deg} deg: n={n}, kappa={kappa:.3f}, qw_kappa={qw:.3f}')
except Exception as exc:
    agreement = {'model_vs_clinician_reliability': {}}
    print(f'[Agreement] error: {exc}')
print('[Agreement] done')


## 7. Hallucination comparison
Log grounded demo output against a deliberately free-form comparison string.

In [ ]:
from app.biocoach_demo import log_hallucination_comparison, run_demo
hallucination_logs = []
try:
    free_form_text = "The knee flexion is 999.0 degrees and trunk lean is 888.0 degrees."
    for detector_record in detector_records[:2]:
        grounded = run_demo(detector_record)
        hallucination_logs.append(log_hallucination_comparison(free_form_text, grounded, detector_record))
except Exception:
    hallucination_logs = []
hallucination_rates = {
    "free_form": float(np.mean([item["free_form_unsupported_count"] for item in hallucination_logs])) if hallucination_logs else 0.0,
    "grounded": float(np.mean([item["grounded_unsupported_count"] for item in hallucination_logs])) if hallucination_logs else 0.0,
}
print("[Hallucination] done")

## 8. Report
Package tables, the risk-coverage figure, manifest, and results bundle.

In [ ]:
import json as _json
from pathlib import Path as _Path
from src.m04_report import (build_results_bundle, build_table1, build_table2,
                            build_table3, build_table4, make_risk_coverage_figure,
                            save_manifest, stamp_synthetic)

try:
    # Use real per-angle rows from the Detector cell
    if not table2_rows:
        table2_rows = [{'angle': a, 'accuracy': detector_metrics.get('accuracy', 0.0)} for a in flags_by_angle]

    table1 = build_table1(data_source, len({s.subject_id for s in samples}), len(samples),
                          detector_metrics.get('accuracy', 0.0), clip_metrics.get('accuracy', 0.0))
    table2 = build_table2(table2_rows)
    table3 = build_table3(detector_metrics.get('ece', 0.0), {}, risk)
    table4 = build_table4(agreement, hallucination_rates)
    table1, table2, table3, table4 = [stamp_synthetic(t, data_is_synthetic) for t in (table1, table2, table3, table4)]

    # Save tables as JSON so they end up in the zip
    _Path('results').mkdir(parents=True, exist_ok=True)
    for name, tbl in [('table1', table1), ('table2', table2), ('table3', table3), ('table4', table4)]:
        _Path(f'results/{name}.json').write_text(_json.dumps(tbl, indent=2, default=str))

    make_risk_coverage_figure(risk.get('coverage', []), risk.get('risk', []))
    save_manifest(data_source, RANDOM_SEED)
    bundle_path = build_results_bundle()

    # Print all tables
    for name in ('table1', 'table2', 'table3', 'table4'):
        content = _json.loads(_Path(f'results/{name}.json').read_text())
        print(f'\n=== {name} ===')
        print(_json.dumps(content, indent=2, default=str))

    # Download
    if os.path.isdir('/content') and not os.path.isdir('/kaggle'):
        try:
            from google.colab import files
            files.download(bundle_path)
        except Exception:
            pass
    elif os.path.isdir('/kaggle'):
        import shutil
        shutil.copy2(bundle_path, '/kaggle/working/results_bundle.zip')
        print(f'[Report] copied to /kaggle/working/results_bundle.zip')
    print(f'[Report] complete: {bundle_path}')
except Exception as exc:
    bundle_path = None
    print(f'[Report] error: {exc}')
    import traceback; traceback.print_exc()
